For each file, find its person id and split it into 5 second chunks

In [1]:
import os
import numpy as np
import pandas as pd
from collections import defaultdict
from pathlib import Path
from types import SimpleNamespace

# ---- Paths ----
kalman_filter_data_path = '../data/kaggle-drdataboston/attempt_3/updated_data_kalman_filtered_3'
matrix_path = '../data/kaggle-drdataboston/matrix.csv'
out_path = '../data/kaggle-drdataboston/attempt_fft'

# ---- Parameters ----
window_ms = 5000
interp_pts = 500

# ---- Load person-session mapping ----
matrix = pd.read_csv(matrix_path)
session_cols = [
    "subject_left_waist_session1", "subject_right_pocket_session1",
    "subject_left_waist_session2", "subject_right_pocket_session2"
]

file_to_person = {}
for _, row in matrix.iterrows():
    person_id = row.iloc[0]  # First column = ID
    for col in session_cols:
        val = row[col]
        if pd.notna(val):
            session_file = str(val).strip().lower()
            session_file = os.path.splitext(session_file)[0]  # remove .csv
            file_to_person[session_file] = person_id

print(f"Total sessions in mapping: {len(file_to_person)}")

# ---- Group files by person ----
person_sessions = defaultdict(list)
unmatched = []

for file in os.listdir(kalman_filter_data_path):
    if not file.endswith('.csv'):
        continue
    # Strip leading/trailing spaces and remove multiple .csv extensions
    name = file.strip().lower()
    while name.endswith('.csv'):
        name = name[:-4]  # remove '.csv'

    person_id = file_to_person.get(name)
    if person_id is not None:
        person_sessions[person_id].append(file)
    else:
        unmatched.append(file)

print(f"Matched people: {len(person_sessions)}")
print(f"Unmatched files: {len(unmatched)}")
if unmatched:
    print("Examples:", unmatched[:5])

# ---- Storage ----
X_train, y_train = [], []
X_val, y_val = [], []
X_test, y_test = [], []

def compute_fft_features(window, max_freq_bins=30):
    """
    Converts a (500, 3) time-series window into a 1D FFT feature vector.
    Returns: shape (max_freq_bins * 3,)
    """
    features = []
    for axis in range(3):
        signal = window[:, axis]
        fft_vals = np.fft.rfft(signal)
        fft_mag = np.abs(fft_vals)
        features.extend(fft_mag[:max_freq_bins])  # Only keep low frequencies
    return np.array(features)


# ---- Helper: Interpolate and extract windows ----
def process_file(file_path, person_id):
    df = pd.read_csv(file_path)
    df['window_id'] = (df['timestamp'] - df['timestamp'].iloc[0]) // window_ms
    chunks = []
    for _, group in df.groupby('window_id'):
        if len(group) < 2:
            continue
        time_arr = group['timestamp'].values - group['timestamp'].values[0]
        new_time = np.linspace(0, time_arr[-1], interp_pts)
        try:
            x = np.interp(new_time, time_arr, group['acc_x_kf'].values)
            y = np.interp(new_time, time_arr, group['acc_y_kf'].values)
            z = np.interp(new_time, time_arr, group['acc_z_kf'].values)
            window = np.stack((x, y, z), axis=-1)  # shape: (500, 3)
            fft_features = compute_fft_features(window)
            chunks.append(fft_features)

        except Exception:
            continue
    return chunks

# ---- Split by session ----
used_people = 0

for person_id, sessions in person_sessions.items():
    if len(sessions) < 3:
        continue

    sessions = sorted(sessions)
    test_file = sessions[0]
    val_file = sessions[1]
    train_files = sessions[2:]

    used_people += 1

    for file in train_files:
        windows = process_file(os.path.join(kalman_filter_data_path, file), person_id)
        X_train.extend(windows)
        y_train.extend([person_id] * len(windows))

    for file in [val_file]:
        windows = process_file(os.path.join(kalman_filter_data_path, file), person_id)
        X_val.extend(windows)
        y_val.extend([person_id] * len(windows))

    for file in [test_file]:
        windows = process_file(os.path.join(kalman_filter_data_path, file), person_id)
        X_test.extend(windows)
        y_test.extend([person_id] * len(windows))

# ---- Save as structured npy files ----
Path(out_path).mkdir(parents=True, exist_ok=True)

dataset_X = SimpleNamespace(train=np.array(X_train), val=np.array(X_val), test=np.array(X_test))
dataset_y = SimpleNamespace(train=np.array(y_train), val=np.array(y_val), test=np.array(y_test))

np.save(f"{out_path}/dataset_X.npy", dataset_X, allow_pickle=True)
np.save(f"{out_path}/dataset_y.npy", dataset_y, allow_pickle=True)

# ---- Print summary ----
def print_split_info(name, X, y):
    print(f"\n{name} set:")
    print(f"  Samples:        {len(y)}")
    print(f"  Unique persons: {len(np.unique(y))}")

print("\n=== Dataset Split Summary ===")
print_split_info("Train", X_train, y_train)
print_split_info("Validation", X_val, y_val)
print_split_info("Test", X_test, y_test)

print(f"\nTotal persons in matrix: {matrix.shape[0]}")
print(f"Persons with ≥3 sessions used: {used_people}")


Total sessions in mapping: 338
Matched people: 92
Unmatched files: 0

=== Dataset Split Summary ===

Train set:
  Samples:        9088
  Unique persons: 62

Validation set:
  Samples:        5935
  Unique persons: 62

Test set:
  Samples:        6020
  Unique persons: 62

Total persons in matrix: 93
Persons with ≥3 sessions used: 62


In [2]:
from collections import Counter

# Count occurrences of each label
label_counts = Counter(y_train)

# Find label with the fewest entries
min_label, min_count = min(label_counts.items(), key=lambda x: x[1])

max_label, max_count = max(label_counts.items(), key=lambda x: x[1])

print(f"\nLabel with fewest samples: {min_label} ({min_count} samples)")
print(f"Label with most samples: {max_label} ({max_count} samples)")



Label with fewest samples: 74 (72 samples)
Label with most samples: 51 (270 samples)


In [3]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input, Flatten
from tensorflow.keras.utils import to_categorical
from sklearn.utils import shuffle


2025-05-02 17:37:53.224447: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-05-02 17:37:53.241030: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-05-02 17:37:53.354840: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-05-02 17:37:53.452988: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1746196673.556083   12733 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1746196673.58

In [4]:
data_path = '../data/kaggle-drdataboston/attempt_fft'
fname = 'dataset'

X_data = np.load(f'{data_path}/{fname}_X.npy', allow_pickle=True).item()  # shape: (N, T)
y_data = np.load(f'{data_path}/{fname}_y.npy', allow_pickle=True).item()  # shape: (N,)

print(X_data.train.shape)
print(y_data.train.shape)



(9088, 90)
(9088,)


In [32]:
# Choose 10 unique person IDs
all_ids = np.unique(y_data.train)
selected_ids = np.random.choice(all_ids, size=10, replace=False)

# Create masks for each split
def filter_by_ids(X, y, ids):
    mask = np.isin(y, ids)
    return X[mask], y[mask]

X_train_10, y_train_10 = filter_by_ids(X_data.train, y_data.train, selected_ids)
X_val_10, y_val_10     = filter_by_ids(X_data.val,   y_data.val,   selected_ids)
X_test_10, y_test_10   = filter_by_ids(X_data.test,  y_data.test,  selected_ids)

print("Train:", X_train_10.shape)
print("Val:  ", X_val_10.shape)
print("Test: ", X_test_10.shape)

Train: (1716, 90)
Val:   (996, 90)
Test:  (984, 90)


In [36]:
import numpy as np
from sklearn.preprocessing import LabelEncoder
import joblib

# Load data
X_data = np.load(f'{data_path}/{fname}_X.npy', allow_pickle=True).item()
y_data = np.load(f'{data_path}/{fname}_y.npy', allow_pickle=True).item()

# Extract full sets
X_train = X_data.train
X_val   = X_data.val
X_test  = X_data.test

y_train_raw = y_data.train
y_val_raw   = y_data.val
y_test_raw  = y_data.test

# 🔹 Choose 10 unique person IDs
all_ids = np.unique(y_train_raw)
selected_ids = np.random.choice(all_ids, size=10, replace=False)

# 🔹 Filter to only those 10
def filter_by_ids(X, y, ids):
    mask = np.isin(y, ids)
    return X[mask], y[mask]

X_train, y_train_raw = filter_by_ids(X_train, y_train_raw, selected_ids)
X_val,   y_val_raw   = filter_by_ids(X_val,   y_val_raw,   selected_ids)
X_test,  y_test_raw  = filter_by_ids(X_test,  y_test_raw,  selected_ids)

# 🔹 Re-encode labels (to range 0–9)
label_encoder = LabelEncoder()
y_all_raw = np.concatenate([y_train_raw, y_val_raw, y_test_raw])
y_all_enc = label_encoder.fit_transform(y_all_raw)

y_train = label_encoder.transform(y_train_raw)
y_val   = label_encoder.transform(y_val_raw)
y_test  = label_encoder.transform(y_test_raw)

# 🔹 Save encoder
joblib.dump(label_encoder, f'{data_path}/label_encoder_10class.pkl')

# ✅ Final check
print("10-person class labels:", label_encoder.classes_)
print("Train shape:", X_train.shape, "→", np.unique(y_train, return_counts=True))


10-person class labels: [ 8 10 15 16 20 23 25 32 34 68]
Train shape: (1550, 90) → (array([0, 1, 2, 3, 4, 5, 6, 7, 8, 9]), array([117, 215, 194, 176, 203, 191, 171,  95,  95,  93]))


In [37]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

joblib.dump(scaler, f'{data_path}/fft_scaler.pkl')


['../data/kaggle-drdataboston/attempt_fft/fft_scaler.pkl']

In [38]:
# Now you can get num_classes safely
num_classes = len(np.unique(y_all_enc))

# One-hot encode
from tensorflow.keras.utils import to_categorical
y_train_cat = to_categorical(y_train, num_classes)
y_val_cat = to_categorical(y_val, num_classes)
y_test_cat = to_categorical(y_test, num_classes)

In [39]:
from sklearn.utils import class_weight
import numpy as np

# Compute weights for each class
class_weights_array = class_weight.compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)

# Convert to dict for Keras
class_weights_dict = dict(enumerate(class_weights_array))


In [40]:
X_train_cnn = X_train_scaled.reshape((X_train_scaled.shape[0], X_train_scaled.shape[1], 1))
X_val_cnn   = X_val_scaled.reshape((X_val_scaled.shape[0], X_val_scaled.shape[1], 1))
X_test_cnn  = X_test_scaled.reshape((X_test_scaled.shape[0], X_test_scaled.shape[1], 1))


In [46]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Input
from tensorflow.keras.optimizers import Adam

model = Sequential([
    Input(shape=(X_train_scaled.shape[1],)),  # e.g., 90

    Dense(32, activation='relu'),
    Dropout(0.1),

    Dense(num_classes, activation='softmax')
])

model.compile(
    optimizer=Adam(learning_rate=1e-4),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()


Model: "sequential_10"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_23 (Dense)                │ (None, 32)             │         2,912 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_26 (Dropout)            │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_24 (Dense)                │ (None, 10)             │           330 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,242 (12.66 KB)

 Trainable params: 3,242 (12.66 KB)

 Non-trainable params: 0 (0.00 B)

In [54]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Conv1D, MaxPooling1D, Flatten, Dense, Dropout
from tensorflow.keras.optimizers import Adam

model2 = Sequential([
    Input(shape=(X_train_scaled.shape[1], 1)),  # e.g., (90, 1)

    Conv1D(16, kernel_size=3, activation='relu', padding='same'),
    MaxPooling1D(pool_size=2),
    Dropout(0.1),

    Conv1D(32, kernel_size=3, activation='relu', padding='same'),
    MaxPooling1D(pool_size=2),
    Dropout(0.1),

    Flatten(),
    Dense(32, activation='relu'),
    Dropout(0.1),

    Dense(num_classes, activation='softmax')
])

model2.compile(
    optimizer=Adam(learning_rate=1e-4),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model2.summary()


Model: "sequential_11"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d_12 (Conv1D)              │ (None, 90, 16)         │            64 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_12 (MaxPooling1D) │ (None, 45, 16)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_27 (Dropout)            │ (None, 45, 16)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_13 (Conv1D)              │ (None, 45, 32)         │         1,568 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_13 (MaxPooling1D) │ (None, 22, 32)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_28 (Dropout)            │ (None, 22, 32)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_6 (Flatten)             │ (None, 704)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_25 (Dense)                │ (None, 32)             │        22,560 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_29 (Dropout)            │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_26 (Dense)                │ (None, 10)             │           330 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 24,522 (95.79 KB)

 Trainable params: 24,522 (95.79 KB)

 Non-trainable params: 0 (0.00 B)

In [55]:
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

early_stop = EarlyStopping(monitor='val_loss', patience=8, restore_best_weights=True)
checkpoint = ModelCheckpoint(f'{out_path}/best_cnn.keras', monitor='val_loss', verbose=1, save_best_only=True, mode='min')


model2.fit(
    X_train_scaled, y_train_cat,
    validation_data=(X_val_scaled, y_val_cat),
    epochs=100,
    batch_size=32,
    verbose=1,
    callbacks = [early_stop, checkpoint],
    class_weight=class_weights_dict
)


Epoch 1/100
45/49 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.1739 - loss: 2.2943
Epoch 1: val_loss improved from inf to 2.27016, saving model to ../data/kaggle-drdataboston/attempt_fft/best_cnn.keras
49/49 ━━━━━━━━━━━━━━━━━━━━ 2s 18ms/step - accuracy: 0.1758 - loss: 2.2919 - val_accuracy: 0.2119 - val_loss: 2.2702
Epoch 2/100
48/49 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.2384 - loss: 2.2097
Epoch 2: val_loss improved from 2.27016 to 2.24983, saving model to ../data/kaggle-drdataboston/attempt_fft/best_cnn.keras
49/49 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - accuracy: 0.2394 - loss: 2.2081 - val_accuracy: 0.2346 - val_loss: 2.2498
Epoch 3/100
48/49 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.3588 - loss: 2.1084
Epoch 3: val_loss improved from 2.24983 to 2.22560, saving model to ../data/kaggle-drdataboston/attempt_fft/best_cnn.keras
49/49 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - accuracy: 0.3592 - loss: 2.1070 - val_accuracy: 0.2428 - val_loss: 2.2256
Epoch 4/100
45/49 ━━━━━━━━━━━━━

In [17]:
from tensorflow.keras.models import load_model

model_lstm = load_model(f'{data_path}/CNN_LSTM/best_cnn_lstm.keras')

In [18]:
test_loss, test_acc = model_lstm.evaluate(X_test_scaled, y_test_cat)
print(f"Test accuracy: {test_acc:.2f}")


123/123 ━━━━━━━━━━━━━━━━━━━━ 7s 52ms/step - accuracy: 0.9607 - loss: 0.1744
Test accuracy: 0.96


In [56]:
test_loss, test_acc = model2.evaluate(X_test_scaled, y_test_cat)
print(f"Test accuracy: {test_acc:.2f}")



31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.1545 - loss: 2.4469
Test accuracy: 0.10
